# Data Validation

Upload a CSV file and validate it against the OpenGLASSLab data schema for glazing data, harvest data, or test data.


<div style="border-left: 5px solid #0074D9; padding: 10px;">
    <strong>Note:</strong> You can use the 🚀 <strong>`Live Code`</strong> button in the top right to activate the interactive features and use Python interactively! <br><br>
                           Once the "Live Code" is enabled it is advised to <strong>"Run All"</strong> cells first to load all the necessary packages and functions. <br><br>
                           Afterwards, upload a CSV file below, select the matching data type from the dropdown, and click <strong>"Validate"</strong> to run the corresponding validation script.
</div>


In [3]:
import tempfile
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, HTML

# Ensure pandas/pydantic are available. Outside a Pyodide (Live Code) sandbox,
# these already come from the local environment and `micropip` simply doesn't
# exist, so this block is a no-op. Inside Pyodide, pydantic is pinned to v2
# explicitly: an unpinned install can silently resolve to an old pure-Python
# pydantic v1 wheel (no C-extension dependency), which lacks APIs such as
# `BeforeValidator` used below (added in pydantic v2). This can't rely on
# Pyodide's auto-import scanner, since the actual `import pydantic` statements
# below are nested inside function bodies, not at the cell's top level.
try:
    import micropip
except ImportError:
    micropip = None

if micropip is not None:
    await micropip.install(["pandas", "pydantic>=2,<3"])

# ---------------------------------------------------------------------------
# Validation logic, inlined from book/scripts/validate_*.py (single source of
# truth: book/scripts/). Each builder is self-contained (its own imports,
# model, and helpers) so names such as `validate_record` or `load_csv` don't
# collide between the three data types. Kept in sync manually with the CLI
# scripts in book/scripts/.
# ---------------------------------------------------------------------------


def _build_glazing_validator():
    """Build and return the `validate_file` function for glazing data CSVs."""
    import re
    from typing import Annotated, Literal

    import pandas as pd
    from pydantic import BaseModel, BeforeValidator, field_validator, model_validator
    from pydantic import ValidationError as PydanticValidationError

    _NULL_STRINGS: set[str] = {"na", "n/a", "-", "", "null"}
    _UNKNOWN_STRINGS: set[str] = _NULL_STRINGS | {"unknown"}

    _BUILDING_ID_PATTERN = re.compile(r"^[A-Z][A-Z0-9\-]*_[A-Z][A-Z0-9\-]*$")
    _BATCH_ID_PATTERN = re.compile(r"^BATCH-\d{2,}$")
    _UNIT_ID_PATTERN = re.compile(r"^UNIT-\d{3,}$")

    def _none_if_null(v):
        if isinstance(v, str) and v.strip().lower() in _UNKNOWN_STRINGS:
            return None
        return v

    def _coerce_int(v):
        if isinstance(v, str):
            try:
                return int(v.strip())
            except ValueError:
                return v
        return v

    def _coerce_float(v):
        if isinstance(v, str):
            try:
                return float(v.strip().replace(",", "."))
            except ValueError:
                return v
        return v

    _NullableStr = Annotated[str | None, BeforeValidator(_none_if_null)]
    _NullableFloat = Annotated[
        float | None, BeforeValidator(_none_if_null), BeforeValidator(_coerce_float)
    ]
    _MandatoryFloat = Annotated[float, BeforeValidator(_coerce_float)]

    class GlazingRecord(BaseModel):
        model_config = {"str_strip_whitespace": True}

        building_id: str
        batch_id: str
        unit_id: str
        unit_width: _MandatoryFloat
        unit_height: _MandatoryFloat
        number_of_panes: Annotated[int, BeforeValidator(_coerce_int)]
        pane_composition_outer: Literal["monolithic", "laminated"]
        pane_composition_inner: Annotated[
            Literal["monolithic", "laminated"] | None, BeforeValidator(_none_if_null)
        ] = None
        pane_composition_middle: Annotated[
            Literal["monolithic", "laminated"] | None, BeforeValidator(_none_if_null)
        ] = None
        pane_glass_type_outer: Literal[
            "annealed", "heat-strengthened", "fully-tempered", "unknown"
        ]
        pane_glass_type_inner: Annotated[
            Literal["annealed", "heat-strengthened", "fully-tempered", "unknown"]
            | None,
            BeforeValidator(_none_if_null),
        ] = None
        pane_glass_type_middle: Annotated[
            Literal["annealed", "heat-strengthened", "fully-tempered", "unknown"]
            | None,
            BeforeValidator(_none_if_null),
        ] = None
        pane_nominal_thickness_outer: _MandatoryFloat
        pane_nominal_thickness_inner: _NullableFloat = None
        pane_nominal_thickness_middle: _NullableFloat = None
        spacer_thickness: _NullableFloat = None
        spacer_material: str
        spacer_code: str
        surface_coating: _NullableStr = None
        notes: _NullableStr = None

        @field_validator("building_id")
        @classmethod
        def check_building_id_format(cls, v):
            if not _BUILDING_ID_PATTERN.match(v):
                raise ValueError(
                    f"building_id '{v}' does not match required format LOCATION_BUILDING (e.g. AMSTERDAM_RIJKSMUSEUM)."
                )
            return v

        @field_validator("batch_id")
        @classmethod
        def check_batch_id_format(cls, v):
            if not _BATCH_ID_PATTERN.match(v):
                raise ValueError(
                    f"batch_id '{v}' does not match required format BATCH-## (e.g. BATCH-01)."
                )
            return v

        @field_validator("unit_id")
        @classmethod
        def check_unit_id_format(cls, v):
            if not _UNIT_ID_PATTERN.match(v):
                raise ValueError(
                    f"unit_id '{v}' does not match required format UNIT-### (e.g. UNIT-001)."
                )
            return v

        @field_validator(
            "unit_width", "unit_height", "pane_nominal_thickness_outer", mode="after"
        )
        @classmethod
        def check_positive_required(cls, v):
            if v <= 0:
                raise ValueError(f"Value must be positive (> 0), got {v}.")
            return v

        @field_validator(
            "pane_nominal_thickness_inner",
            "pane_nominal_thickness_middle",
            mode="after",
        )
        @classmethod
        def check_positive_optional(cls, v):
            if v is not None and v <= 0:
                raise ValueError(f"Value must be positive (> 0) when provided, got {v}.")
            return v

        @field_validator("number_of_panes", mode="after")
        @classmethod
        def check_number_of_panes_positive(cls, v):
            if v < 1:
                raise ValueError(f"number_of_panes must be a positive integer, got {v}.")
            return v

        @field_validator("spacer_thickness", mode="after")
        @classmethod
        def check_non_negative(cls, v):
            if v is not None and v < 0:
                raise ValueError(f"Value must be non-negative, got {v}.")
            return v

        @model_validator(mode="after")
        def check_pane_fields_consistency(self):
            n = self.number_of_panes

            if n >= 2:
                missing = []
                if self.pane_composition_inner is None:
                    missing.append("pane_composition_inner")
                if self.pane_glass_type_inner is None:
                    missing.append("pane_glass_type_inner")
                if self.pane_nominal_thickness_inner is None:
                    missing.append("pane_nominal_thickness_inner")
                if missing:
                    raise ValueError(
                        f"number_of_panes is {n} but inner pane fields are missing: "
                        f"{', '.join(missing)}. Fill these fields or use 'n/a' only "
                        f"for single-pane units."
                    )

            if n >= 3:
                missing = []
                if self.pane_composition_middle is None:
                    missing.append("pane_composition_middle")
                if self.pane_glass_type_middle is None:
                    missing.append("pane_glass_type_middle")
                if self.pane_nominal_thickness_middle is None:
                    missing.append("pane_nominal_thickness_middle")
                if missing:
                    raise ValueError(
                        f"number_of_panes is {n} but middle pane fields are missing: "
                        f"{', '.join(missing)}. Fill these fields or use 'n/a' only "
                        f"for single and double-pane units."
                    )

            return self

    class GlazingDataError(Exception):
        def __init__(self, row, field, message, severity="ERROR"):
            self.row = row
            self.field = field
            self.message = message
            self.severity = severity
            super().__init__(str(self))

        def __str__(self):
            row_part = f"row {self.row}" if self.row is not None else "file level"
            return f"[{self.severity}] In column '{self.field}' at {row_part}: {self.message}"

    def _required_columns_from_model():
        return set(GlazingRecord.model_fields.keys())

    def validate_record(row_index, row):
        errors = []

        try:
            record = GlazingRecord(**row)
        except PydanticValidationError as exc:
            for err in exc.errors():
                field = str(err["loc"][0]) if err["loc"] else "unknown"
                errors.append(GlazingDataError(row_index, field, err["msg"]))
            return errors

        if record.number_of_panes == 1:
            unexpected = []
            if record.pane_composition_inner is not None:
                unexpected.append("pane_composition_inner")
            if record.pane_glass_type_inner is not None:
                unexpected.append("pane_glass_type_inner")
            if record.pane_nominal_thickness_inner is not None:
                unexpected.append("pane_nominal_thickness_inner")
            if record.pane_composition_middle is not None:
                unexpected.append("pane_composition_middle")
            if record.pane_glass_type_middle is not None:
                unexpected.append("pane_glass_type_middle")
            if record.pane_nominal_thickness_middle is not None:
                unexpected.append("pane_nominal_thickness_middle")
            if unexpected:
                errors.append(
                    GlazingDataError(
                        row_index,
                        "pane fields",
                        f"number_of_panes is 1 but inner/middle pane fields are filled: "
                        f"{', '.join(unexpected)}. These should be 'n/a' for single-pane units.",
                        severity="WARNING",
                    )
                )

        if record.number_of_panes >= 2 and record.spacer_thickness is None:
            errors.append(
                GlazingDataError(
                    row_index,
                    "spacer_thickness",
                    f"number_of_panes is {record.number_of_panes} but spacer_thickness is "
                    f"missing. Provide the spacer width in millimetres.",
                    severity="WARNING",
                )
            )

        return errors

    def load_csv(filepath):
        for encoding in ("utf-8-sig", "cp1252"):
            try:
                df = pd.read_csv(
                    filepath,
                    sep=None,
                    engine="python",
                    encoding=encoding,
                    dtype=str,
                    keep_default_na=False,
                )
                break
            except UnicodeDecodeError:
                continue
        else:
            raise ValueError(
                f"Could not decode {filepath} with any supported encoding (utf-8, cp1252)."
            )

        df.columns = [c.strip() for c in df.columns]
        df = df.apply(lambda col: col.map(lambda v: "" if pd.isna(v) else str(v).strip()))
        return df

    def validate_file(filepath):
        print(f"\n{'=' * 60}")
        print(f"Validating: {filepath}")
        print(f"{'=' * 60}")

        try:
            df = load_csv(filepath)
        except Exception as exc:
            print(f"[ERROR] File | Could not read file: {exc}")
            return False

        all_errors = []

        required = _required_columns_from_model()
        missing = required - set(df.columns)
        for col in sorted(missing):
            all_errors.append(
                GlazingDataError(
                    None, col, f"Required column '{col}' is missing from the file."
                )
            )

        rows = df.to_dict(orient="records")

        for idx, row in enumerate(rows, start=1):
            all_errors.extend(validate_record(idx, row))

        seen_ids = {}
        for idx, row in enumerate(rows, start=1):
            bid = str(row.get("building_id", "")).strip()
            uid = str(row.get("unit_id", "")).strip()
            combo = f"{bid}|{uid}"
            if bid and uid and bid.lower() not in _UNKNOWN_STRINGS:
                if combo in seen_ids:
                    all_errors.append(
                        GlazingDataError(
                            idx,
                            "building_id + unit_id",
                            f"Duplicate combination building_id='{bid}', unit_id='{uid}' "
                            f"(first seen at row {seen_ids[combo]}).",
                        )
                    )
                else:
                    seen_ids[combo] = idx

        errors = [e for e in all_errors if e.severity == "ERROR"]
        warnings = [e for e in all_errors if e.severity == "WARNING"]

        if not all_errors:
            print(f"All checks passed. ({len(rows)} row(s) validated)")
            return True

        for e in sorted(all_errors, key=lambda x: (x.row or 0, x.severity)):
            print(e)

        print(
            f"\nSummary: {len(rows)} row(s) | "
            f"{len(errors)} error(s) | {len(warnings)} warning(s)"
        )
        return len(errors) == 0

    return validate_file


def _build_harvest_validator():
    """Build and return the `validate_file` function for harvest data CSVs."""
    import re
    from datetime import date
    from typing import Annotated, Literal

    import pandas as pd
    from pydantic import BaseModel, BeforeValidator, field_validator, model_validator
    from pydantic import ValidationError as PydanticValidationError

    _NULL_STRINGS: set[str] = {"na", "n/a", "-", "", "null"}
    _UNKNOWN_STRINGS: set[str] = _NULL_STRINGS | {"unknown"}

    _MIN_YEAR: int = 1800
    _MAX_YEAR: int = date.today().year

    _BUILDING_ID_PATTERN = re.compile(r"^[A-Z][A-Z0-9\-]*_[A-Z][A-Z0-9\-]*$")
    _BATCH_ID_PATTERN = re.compile(r"^BATCH-\d{2,}$")

    def _none_if_null(v):
        if isinstance(v, str) and v.strip().lower() in _UNKNOWN_STRINGS:
            return None
        return v

    def _coerce_int(v):
        if isinstance(v, str):
            try:
                return int(v.strip())
            except ValueError:
                return v
        return v

    _NullableInt = Annotated[
        int | None, BeforeValidator(_none_if_null), BeforeValidator(_coerce_int)
    ]
    _NullableStr = Annotated[str | None, BeforeValidator(_none_if_null)]

    class HarvestRecord(BaseModel):
        model_config = {"str_strip_whitespace": True}

        building_id: str
        batch_id: str
        building_country: str
        building_city: str
        building_street: _NullableStr = None
        building_coordinates: _NullableStr = None
        building_use: Literal[
            "residential",
            "office",
            "industrial",
            "commercial",
            "institutional",
            "mixed-use",
        ]
        year_of_installation: _NullableInt = None
        year_of_harvesting: Annotated[int, BeforeValidator(_coerce_int)]
        years_in_service: _NullableInt = None
        glazing_type: Literal["single", "double", "triple", "other"]
        building_element_type: Literal[
            "window",
            "door",
            "roof/skylight",
            "partition wall",
            "facade/curtain wall",
            "floor",
            "balustrade",
            "canopy",
            "other",
        ]
        exposure_type: Literal["exterior", "interior"]
        facade_orientation: Literal[
            "north",
            "south",
            "east",
            "west",
            "northeast",
            "northwest",
            "southeast",
            "southwest",
            "horizontal",
            "unknown",
        ]
        floor_level: _NullableInt = None
        harvester: _NullableStr = None
        harvest_date: Annotated[date | None, BeforeValidator(_none_if_null)] = None
        notes: _NullableStr = None

        @field_validator("building_id")
        @classmethod
        def check_building_id_format(cls, v):
            if not _BUILDING_ID_PATTERN.match(v):
                raise ValueError(
                    f"building_id '{v}' does not match required format LOCATION_BUILDING (e.g. AMSTERDAM_RIJKSMUSEUM)."
                )
            return v

        @field_validator("batch_id")
        @classmethod
        def check_batch_id_format(cls, v):
            if not _BATCH_ID_PATTERN.match(v):
                raise ValueError(
                    f"batch_id '{v}' does not match required format BATCH-## (e.g. BATCH-01)."
                )
            return v

        @field_validator("year_of_installation", "year_of_harvesting", mode="after")
        @classmethod
        def check_year_range(cls, v):
            if v is not None and not (_MIN_YEAR <= v <= _MAX_YEAR):
                raise ValueError(
                    f"Year {v} is outside plausible range [{_MIN_YEAR}\u2013{_MAX_YEAR}]."
                )
            return v

        @model_validator(mode="after")
        def check_years_in_service(self):
            yi = self.year_of_installation
            yh = self.year_of_harvesting
            yis = self.years_in_service

            if yi is None and yis is None:
                raise ValueError(
                    "At least one of year_of_installation or years_in_service must be "
                    "provided; the other can be derived from year_of_harvesting."
                )

            if yi is not None and yh is not None:
                if yh < yi:
                    raise ValueError(
                        f"year_of_harvesting ({yh}) is before year_of_installation ({yi})."
                    )
                expected = yh - yi
                if yis is not None and yis != expected:
                    raise ValueError(
                        f"years_in_service ({yis}) does not match "
                        f"year_of_harvesting - year_of_installation "
                        f"({yh} - {yi} = {expected})."
                    )
            return self

    class HarvestDataError(Exception):
        def __init__(self, row, field, message, severity="ERROR"):
            self.row = row
            self.field = field
            self.message = message
            self.severity = severity
            super().__init__(str(self))

        def __str__(self):
            row_part = f"row {self.row}" if self.row is not None else "file level"
            return f"[{self.severity}] In column '{self.field}' at {row_part}: {self.message}"

    def _required_columns_from_model():
        return set(HarvestRecord.model_fields.keys())

    def validate_record(row_index, row):
        errors = []

        try:
            record = HarvestRecord(**row)
        except PydanticValidationError as exc:
            for err in exc.errors():
                field = str(err["loc"][0]) if err["loc"] else "unknown"
                errors.append(HarvestDataError(row_index, field, err["msg"]))
            return errors

        if record.harvest_date is not None and record.harvest_date > date.today():
            errors.append(
                HarvestDataError(
                    row_index,
                    "harvest_date",
                    f"harvest_date '{record.harvest_date}' is in the future.",
                    severity="WARNING",
                )
            )

        if record.years_in_service is None and record.year_of_installation is not None:
            derived = record.year_of_harvesting - record.year_of_installation
            errors.append(
                HarvestDataError(
                    row_index,
                    "years_in_service",
                    f"Field is empty but can be derived: "
                    f"{record.year_of_harvesting} - {record.year_of_installation} = {derived}.",
                    severity="WARNING",
                )
            )

        return errors

    def load_csv(filepath):
        for encoding in ("utf-8-sig", "cp1252"):
            try:
                df = pd.read_csv(
                    filepath,
                    sep=None,
                    engine="python",
                    encoding=encoding,
                    dtype=str,
                    keep_default_na=False,
                )
                break
            except UnicodeDecodeError:
                continue
        else:
            raise ValueError(
                f"Could not decode {filepath} with any supported encoding (utf-8, cp1252)."
            )

        df.columns = [c.strip() for c in df.columns]
        df = df.apply(lambda col: col.map(lambda v: "" if pd.isna(v) else str(v).strip()))
        return df

    def validate_file(filepath):
        print(f"\n{'=' * 60}")
        print(f"Validating: {filepath}")
        print(f"{'=' * 60}")

        try:
            df = load_csv(filepath)
        except Exception as exc:
            print(f"[ERROR] File | Could not read file: {exc}")
            return False

        all_errors = []

        required = _required_columns_from_model()
        missing = required - set(df.columns)
        for col in sorted(missing):
            all_errors.append(
                HarvestDataError(
                    None, col, f"Required column '{col}' is missing from the file."
                )
            )

        rows = df.to_dict(orient="records")

        for idx, row in enumerate(rows, start=1):
            all_errors.extend(validate_record(idx, row))

        seen_ids = {}
        for idx, row in enumerate(rows, start=1):
            bid = str(row.get("building_id", "")).strip()
            batchid = str(row.get("batch_id", "")).strip()
            combo = f"{bid}|{batchid}"
            if bid and batchid and bid.lower() not in _UNKNOWN_STRINGS:
                if combo in seen_ids:
                    all_errors.append(
                        HarvestDataError(
                            idx,
                            "building_id + batch_id",
                            f"Duplicate combination building_id='{bid}', batch_id='{batchid}' "
                            f"(first seen at row {seen_ids[combo]}).",
                        )
                    )
                else:
                    seen_ids[combo] = idx

        errors = [e for e in all_errors if e.severity == "ERROR"]
        warnings = [e for e in all_errors if e.severity == "WARNING"]

        if not all_errors:
            print(f"All checks passed. ({len(rows)} row(s) validated)")
            return True

        for e in sorted(all_errors, key=lambda x: (x.row or 0, x.severity)):
            print(e)

        print(
            f"\nSummary: {len(rows)} row(s) | "
            f"{len(errors)} error(s) | {len(warnings)} warning(s)"
        )
        return len(errors) == 0

    return validate_file


def _build_test_validator():
    """Build and return the `validate_file` function for test data CSVs."""
    import re
    from typing import Annotated, Literal

    import pandas as pd
    from pydantic import BaseModel, BeforeValidator, field_validator
    from pydantic import ValidationError as PydanticValidationError

    _NULL_STRINGS: set[str] = {"na", "n/a", "-", "", "null"}
    _UNKNOWN_STRINGS: set[str] = _NULL_STRINGS | {"unknown"}

    _BUILDING_ID_PATTERN = re.compile(r"^[A-Z][A-Z0-9\-]*_[A-Z][A-Z0-9\-]*$")
    _BATCH_ID_PATTERN = re.compile(r"^BATCH-\d{2,}$")
    _UNIT_ID_PATTERN = re.compile(r"^UNIT-\d{3,}$")
    _SPECIMEN_ID_PATTERN = re.compile(r"^SPEC-\d{3,}$")
    _DATE_TIME_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")

    def _none_if_null(v):
        if isinstance(v, str) and v.strip().lower() in _UNKNOWN_STRINGS:
            return None
        return v

    def _coerce_float(v):
        if isinstance(v, str):
            try:
                return float(v.strip().replace(",", "."))
            except ValueError:
                return v
        return v

    _NullableStr = Annotated[str | None, BeforeValidator(_none_if_null)]
    _NullableFloat = Annotated[
        float | None, BeforeValidator(_none_if_null), BeforeValidator(_coerce_float)
    ]
    _MandatoryFloat = Annotated[float, BeforeValidator(_coerce_float)]

    class TestRecord(BaseModel):
        model_config = {"str_strip_whitespace": True}

        building_id: str
        batch_id: str
        unit_id: str
        specimen_id: str
        specimen_pretreatment: _NullableStr = None
        specimen_predetection: Annotated[
            Literal[
                "not inspected",
                "none detected",
                "detected - naked eye",
                "detected - microscope",
                "detected - scanner",
                "unknown",
            ]
            | None,
            BeforeValidator(_none_if_null),
        ] = None
        surface_id: Literal["S1", "S2", "S3", "S4", "S5", "S6"]
        air_or_tin: Literal["air", "tin", "unknown", "n/a"]
        prestress_state: Literal[
            "annealed", "heat-strengthened", "fully-tempered", "unknown"
        ]
        prestress_measured: _NullableFloat = None
        prestress_device: _NullableStr = None
        specimen_shape: Literal["square", "circular"]
        specimen_edgelength: _NullableFloat = None
        specimen_diameter: _MandatoryFloat
        thickness_nominal: _MandatoryFloat
        thickness_measured: _MandatoryFloat
        specimen_code: _NullableStr = None
        diameter_supportring: _MandatoryFloat
        diameter_loadring: _MandatoryFloat
        rings_standard: Literal["ASTM", "CEN", "none"]
        target_stressrate: _MandatoryFloat
        machine_control: Literal[
            "force-controlled-kns", "displacement-controlled-mms"
        ]
        target_machinerate: _MandatoryFloat
        fracture_location: Literal[
            "inside the load ring",
            "at the load ring",
            "outside the load ring",
            "at the support ring",
            "outside the support ring",
            "at the edge",
            "unknown",
        ]
        fracture_notes: _NullableStr = None
        fracture_load: _MandatoryFloat
        fracture_time: _MandatoryFloat
        true_loadrate: _MandatoryFloat
        true_stressrate: _MandatoryFloat
        temperature: _NullableFloat = None
        humidity: _NullableFloat = None
        date_time: str
        fracture_stress: _MandatoryFloat
        fracture_stress_equivalent_5s: _NullableFloat = None
        fracture_stress_equivalent_60s: _NullableFloat = None

        @field_validator("building_id")
        @classmethod
        def check_building_id_format(cls, v):
            if not _BUILDING_ID_PATTERN.match(v):
                raise ValueError(
                    f"building_id '{v}' does not match required format LOCATION_BUILDING (e.g. AMSTERDAM_RIJKSMUSEUM)."
                )
            return v

        @field_validator("batch_id")
        @classmethod
        def check_batch_id_format(cls, v):
            if not _BATCH_ID_PATTERN.match(v):
                raise ValueError(
                    f"batch_id '{v}' does not match required format BATCH-## (e.g. BATCH-01)."
                )
            return v

        @field_validator("unit_id")
        @classmethod
        def check_unit_id_format(cls, v):
            if not _UNIT_ID_PATTERN.match(v):
                raise ValueError(
                    f"unit_id '{v}' does not match required format UNIT-### (e.g. UNIT-001)."
                )
            return v

        @field_validator("specimen_id")
        @classmethod
        def check_specimen_id_format(cls, v):
            if not _SPECIMEN_ID_PATTERN.match(v):
                raise ValueError(
                    f"specimen_id '{v}' does not match required format SPEC-### (e.g. SPEC-001)."
                )
            return v

        @field_validator("date_time")
        @classmethod
        def check_date_time_format(cls, v):
            if not _DATE_TIME_PATTERN.match(v):
                raise ValueError(
                    f"date_time '{v}' does not match required format YYYY-MM-DD hh:mm:ss (e.g. 2026-10-23 10:03:35)."
                )
            return v

        @field_validator("fracture_stress", mode="after")
        @classmethod
        def check_positive_required(cls, v):
            if v <= 0:
                raise ValueError(f"Value must be positive (> 0), got {v}.")
            return v

        @field_validator(
            "fracture_stress_equivalent_5s",
            "fracture_stress_equivalent_60s",
            mode="after",
        )
        @classmethod
        def check_positive_optional(cls, v):
            if v is not None and v <= 0:
                raise ValueError(f"Value must be positive (> 0) when provided, got {v}.")
            return v

        @field_validator(
            "specimen_edgelength",
            "specimen_diameter",
            "thickness_nominal",
            "thickness_measured",
            "diameter_supportring",
            "diameter_loadring",
            "target_stressrate",
            "target_machinerate",
            "fracture_load",
            "fracture_time",
            "true_loadrate",
            "true_stressrate",
            "humidity",
            mode="after",
        )
        @classmethod
        def check_non_negative(cls, v):
            if v is not None and v < 0:
                raise ValueError(f"Value must be non-negative, got {v}.")
            return v

    class TestDataError(Exception):
        def __init__(self, row, field, message, severity="ERROR"):
            self.row = row
            self.field = field
            self.message = message
            self.severity = severity
            super().__init__(str(self))

        def __str__(self):
            row_part = f"row {self.row}" if self.row is not None else "file level"
            return f"[{self.severity}] In column '{self.field}' at {row_part}: {self.message}"

    def _required_columns_from_model():
        return set(TestRecord.model_fields.keys())

    def validate_record(row_index, row):
        errors = []

        try:
            TestRecord(**row)
        except PydanticValidationError as exc:
            for err in exc.errors():
                field = str(err["loc"][0]) if err["loc"] else "unknown"
                errors.append(TestDataError(row_index, field, err["msg"]))
            return errors

        return errors

    def load_csv(filepath):
        for encoding in ("utf-8-sig", "cp1252"):
            try:
                df = pd.read_csv(
                    filepath,
                    sep=None,
                    engine="python",
                    encoding=encoding,
                    dtype=str,
                    keep_default_na=False,
                )
                break
            except UnicodeDecodeError:
                continue
        else:
            raise ValueError(
                f"Could not decode {filepath} with any supported encoding (utf-8, cp1252)."
            )

        df.columns = [c.strip() for c in df.columns]
        df = df.apply(lambda col: col.map(lambda v: "" if pd.isna(v) else str(v).strip()))
        return df

    def validate_file(filepath):
        print(f"\n{'=' * 60}")
        print(f"Validating: {filepath}")
        print(f"{'=' * 60}")

        try:
            df = load_csv(filepath)
        except Exception as exc:
            print(f"[ERROR] File | Could not read file: {exc}")
            return False

        all_errors = []

        required = _required_columns_from_model()
        missing = required - set(df.columns)
        for col in sorted(missing):
            all_errors.append(
                TestDataError(
                    None, col, f"Required column '{col}' is missing from the file."
                )
            )

        rows = df.to_dict(orient="records")

        for idx, row in enumerate(rows, start=1):
            all_errors.extend(validate_record(idx, row))

        seen_ids = {}
        for idx, row in enumerate(rows, start=1):
            bid = str(row.get("building_id", "")).strip()
            batchid = str(row.get("batch_id", "")).strip()
            uid = str(row.get("unit_id", "")).strip()
            sid = str(row.get("specimen_id", "")).strip()
            combo = f"{bid}|{batchid}|{uid}|{sid}"
            if bid and batchid and uid and sid and bid.lower() not in _UNKNOWN_STRINGS:
                if combo in seen_ids:
                    all_errors.append(
                        TestDataError(
                            idx,
                            "building_id + batch_id + unit_id + specimen_id",
                            f"Duplicate combination building_id='{bid}', batch_id='{batchid}', "
                            f"unit_id='{uid}', specimen_id='{sid}' (first seen at row {seen_ids[combo]}).",
                        )
                    )
                else:
                    seen_ids[combo] = idx

        errors = [e for e in all_errors if e.severity == "ERROR"]
        warnings = [e for e in all_errors if e.severity == "WARNING"]

        if not all_errors:
            print(f"All checks passed. ({len(rows)} row(s) validated)")
            return True

        for e in sorted(all_errors, key=lambda x: (x.row or 0, x.severity)):
            print(e)

        print(
            f"\nSummary: {len(rows)} row(s) | "
            f"{len(errors)} error(s) | {len(warnings)} warning(s)"
        )
        return len(errors) == 0

    return validate_file


# Maps each dropdown option to the matching `validate_file` function
VALIDATORS = {
    "Glazing data": _build_glazing_validator(),
    "Harvesting data": _build_harvest_validator(),
    "Test data": _build_test_validator(),
}


In [ ]:
# Create an HTML widget for instructions
instructions = widgets.HTML(
    value="""
    <p style="font-size: 16px">Upload a CSV file, select the data type it contains, then click <b>Validate</b>.</p>
    <b style="font-size: 16px">The uploaded file is processed locally and is not transmitted over the network or saved permanently.</b>
    """
)

# Widget to upload a CSV file
upload_widget = widgets.FileUpload(
    description="Upload CSV",
    accept=".csv",
    multiple=False,
    layout=widgets.Layout(width="250px"),
)

# Dropdown to select which validation script should be run
data_type_dropdown = widgets.Dropdown(
    options=["Glazing data", "Harvesting data", "Test data"],
    description="Data type:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="300px"),
)

# Button to trigger validation
validate_button = widgets.Button(
    description="Validate",
    button_style="success",
    icon="check",
)

# Output container for the validation report
output = widgets.Output()


def _get_uploaded_file():
    """Return (filename, content_bytes) for the uploaded file, or (None, None)."""
    value = upload_widget.value
    if not value:
        return None, None
    if isinstance(value, dict):
        # ipywidgets 7.x API: {filename: {"content": ..., "metadata": ...}}
        filename, info = next(iter(value.items()))
    else:
        # ipywidgets 8.x API: tuple of dicts with "name" and "content"
        info = value[0]
        filename = info["name"]
    return filename, bytes(info["content"])


def on_validate_clicked(_):
    output.clear_output()
    with output:
        filename, content = _get_uploaded_file()
        if filename is None:
            print("Please upload a CSV file before clicking Validate.")
            return

        validator = VALIDATORS[data_type_dropdown.value]

        with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
            tmp.write(content)
            tmp_path = Path(tmp.name)

        print(f"Uploaded file: {filename}")
        print(f"Validating as: {data_type_dropdown.value}\n")

        try:
            passed = validator(tmp_path)
        finally:
            tmp_path.unlink(missing_ok=True)

        if passed:
            display(HTML('<p style="color: green; font-weight: bold;">✅ Validation passed.</p>'))
        else:
            display(HTML('<p style="color: red; font-weight: bold;">❌ Validation failed. See details above.</p>'))


validate_button.on_click(on_validate_clicked)

display(
    instructions,
    widgets.VBox([widgets.HBox([upload_widget, data_type_dropdown, validate_button]), output]),
)


HTML(value='\n    <p style="font-size: 16px">Upload a CSV file, select the data type it contains, then click <…